In [1]:
#|default_exp client

# Client & Chat

In [2]:
#| hide
from nbdev.showdoc import *

## Setup

In [3]:
#| exports

from gaspare.utils import *
from gaspare.core import *

import os



from typing import Union
from io import BytesIO
from functools import wraps
from google import genai
from google.genai import types

from fastcore.all import *
from fastcore.docments import *

from toolslm.funccall import call_func


In [25]:
c = genai.client.Client(api_key=os.environ.get("GEMINI_API_KEY"))

## Tool prepping

There are [two ways](https://googleapis.github.io/python-genai/#function-calling) of managing function calling. The other is to actually build the function declaration. The first approach has the advantage of enabling automatic function calling, meaning that whenever the LLM decides that the function needs to be called, it will call and get the result back. The main drawback is that it realies on the `types.FunctionDeclaration.from_callable` method, which is quite limited (mainly, it does not add descriptions to the parameters, most likely relying on docstrings following Google's [style guide](https://google.github.io/styleguide/pyguide.html#383-functions-and-methods)).

The second approach requires manually declaring the function, but this won't be picked up by the automatic function calling, so requires manually enabling the function loop (extracting the function calls from the response, calling the function and passing it back to the LLM).

In [5]:
def add1(
    a:int, # the 1st number to add
    b=0,   # the 2nd number to add
)->int:    # the result of adding `a` to `b`
    "Sums two numbers."
    return a+b

def add2(
    a:int, # the 1st number to add
    b:int=0,   # the 2nd number to add
)->int:    # the result of adding `a` to `b`
    "Sums two numbers."
    return a+b

def add3(
    a:int, # the 1st number to add
    b:int,   # the 2nd number to add
)->int:    # the result of adding `a` to `b`
    "Sums two numbers."
    return a+b

try:
    f_decl = types.FunctionDeclaration.from_callable(callable=add1, client=c)
except:
    try:
        f_decl = types.FunctionDeclaration.from_callable(callable=add2, client=c)
    except:
        f_decl = types.FunctionDeclaration.from_callable(callable=add3, client=c)
        

f_decl.to_json_dict()

{'description': 'Sums two numbers.',
 'name': 'add3',
 'parameters': {'properties': {'a': {'type': 'INTEGER'},
   'b': {'type': 'INTEGER'}},
  'type': 'OBJECT'}}

In [6]:
docments(add1, full=True, returns=False)

```json
{ 'a': { 'anno': <class 'int'>,
         'default': <class 'inspect._empty'>,
         'docment': 'the 1st number to add'},
  'b': { 'anno': <class 'int'>,
         'default': 0,
         'docment': 'the 2nd number to add'}}
```

Notice that `types.FunctionDeclaration.from_callable`:

1. Cannot infer the parameter type from the default value (while docments can)
2. Does not use the default values at all: in fact adding default values to a function declaration passed to the Gemini API will raise an error, although the LLM would be able to use (and in fact does use if it's passed), the "required" field of the "parameters", which again could be inferred.

In [7]:
#| export

def _googlify_docs(fdoc:str,                  # Docstring of a function
                    argdescs: dict|None=None, # Dict of arg:docment of the arguments of the function
                    retd: str|None=None       # Return docoment of the function
                   )-> str: # The function docstring following Google style guide
    """Turns a function docment and docstring into a docstrings that function """
    if argdescs: fdoc += "\n\nArgs:\n"  + "\n".join([f"    {p}: {desc}" for p, desc in argdescs.items()])
    if retd: fdoc += f"\n\nReturns:\n    {retd}"
    return fdoc

def goog_doc(f:callable # A docment style function
            )->str:     # Google style docstring
    """Builds the docstring for a docment style function following Google style guide"""
    fdoc = f.__doc__
    args = {par: doc for par, doc in docments(f, returns=False).items() if doc is not None}
    retd = docments(f, full=True, returns=True)['return']['docment']
    return _googlify_docs(fdoc, args, retd)
    

When passing a function as a tool, the API will only access the function signature and docstring to build the `FunctionDeclaration`, so unless they are part of the docstring, the LLM has no way of knowing what the arguments or the returned value are. Assuming that Gemini models will have seen quite a lot of google code, and considering the examples in the documentation, it's probably a good idea to turn the docstrings of tools into a format compatible with the style guide.

In [8]:
print(goog_doc(goog_doc))

Builds the docstring for a docment style function following Google style guide

Args:
    f: A docment style function

Returns:
    Google style docstring


In [9]:
#| exports

def _geminify(f: callable) -> callable:
    """Makes a function suitable to be turned into a function declaration: 
    infers argument types from default values and removes the values from the signature"""
    docs = docments(f, full=True)
    new_params = [inspect.Parameter(name=n,
                                    kind=inspect.Parameter.POSITIONAL_OR_KEYWORD,
                                    annotation=i.anno) for n, i in docs.items() if n != 'return']

        
    @wraps(f)
    def wrapper(*args, **kwargs):
        return f(*args, **kwargs)
    
    wrapper.__signature__ = inspect.Signature(new_params, return_annotation=docs['return']['anno'])
    wrapper.__annotations__ = {n: i['anno'] for n, i in docs.items() if n != 'return'}
    return wrapper


def prep_tool(f:callable, # The function to be passed to the LLM
             as_decl:bool=False,  # Return an enriched genai.types.FunctionDeclaration?
             googlify_docstring:bool=True): # Use docments to rewrite the docstring following Google Style Guide? 
    """Optimizes a dunction for function calling with the Gemini api. Best suited for docments style functions."""
    docs = goog_doc(f) if googlify_docstring else f.__doc__
    _f = _geminify(f)
    _f.__doc__ = docs
    if not as_decl: return _f
    f_decl = types.FunctionDeclaration.from_callable_with_api_option(callable=_f, api_option='GEMINI_API')
    for par, desc in docments(f, returns=False).items():
        if desc: f_decl.parameters.properties[par].description = desc
    required_params = [p for p, d in docments(f, full=True, returns=False).items() if d['default'] == inspect._empty]
    if getattr(f_decl.parameters, "required", None): f_decl.parameters.required = required_params
    return f_decl

To prepare a function to be used as a function declaration, it needs to be stripped of default values and all the arguments need to be annotated. Turning the docstrings in a Google compatible format makes sure that the result can be used for automatic function calling. We rely on `FunctionDeclaration.from_callable` either implicitly (when passing the prepped function to the LLM) or implicitly to do the necessary type conversions of the annotations (i.e. turning a `float` into `NUMBER`, a `str` into `STRING` etc.). If building the function declaration explicitly, we can also enrich it with information from the original function (namely the presence of default values and the arguments docments that can be added paramters objects).

In [10]:
x = prep_tool(add1, as_decl=True)
x

<ul><li><code>description</code>: Sums two numbers.

Args:
    a: the 1st number to add
    b: the 2nd number to add

Returns:
    the result of adding `a` to `b`</li><li><code>name</code>: add1</li><li><code>parameters</code>: <ul><li><code>type</code>: Type.OBJECT</li><li><code>properties</code>: <ul><li><b>a</b>: <ul><li><code>type</code>: Type.INTEGER</li><li><code>description</code>: the 1st number to add</li></ul></li>
<li><b>b</b>: <ul><li><code>type</code>: Type.INTEGER</li><li><code>description</code>: the 2nd number to add</li></ul></li></ul></li></ul></li></ul>

In [27]:
def g(client, inps, model, sp='', temp=0.6, maxtok=None, stream=False, stop=None, **kwargs):
    config = client.models._genconf(sp=sp, temp=temp, maxtok=maxtok, stop=stop, model=model, use_afc=False, **kwargs)
    contents = mk_contents(inps, cli=client)
    return client.models._gen(inps, model, config, stream)

model = models[0]
model

'gemini-2.0-flash'

In [28]:
tool = types.Tool(function_declarations=[x])

a,b = 694599,645893212
pr = f"What is {a}+{b}?"


respt = g(c, pr, model='gemini-2.0-flash', tools=[tool])
respt

<ul><li><code>add1(b=645893212, a=694599)</code></li></ul>
<details><ul><li><code>automatic_function_calling_history</code>: </li><li><code>usage_metadata</code>: Cached: 0; In: 83; Out: 4; Total: 87</li><li><code>candidates</code>: <details open='true'><summary>candidates[0]</summary><ul><li><code>avg_logprobs</code>: 9.5092982519418e-06</li><li><code>finish_reason</code>: FinishReason.STOP</li><li><code>content</code>: <ul><li><code>parts</code>: <details open='true'><summary>parts[0]</summary><ul><li><code>function_call</code>: <ul><li><code>args</code>: <ul><li><b>b</b>: 645893212</li>
<li><b>a</b>: 694599</li></ul></li><li><code>name</code>: add1</li></ul></li></ul></details></li><li><code>role</code>: model</li></ul></li></ul></details></li><li><code>model_version</code>: gemini-2.0-flash</li></ul></details>

In [35]:
#| exports

def f_result(fname, fargs, ns=None):
    try: return {"result": call_func(fname, fargs, ns or globals())}
    except Exception as e: return {'error': str(e)}

def f_results(fcalls, ns=None):
    return [{"name": c.name, "response": f_result(c.name, c.args, ns)} for c in fcalls]

def mk_fres_content(fres):
    return types.Content(role='tool', parts=[types.Part.from_function_response(**d) for d in fres])

These three function are more or less what the SDK automatic function calling does under the hood, turning the function call response form the LLM into an actual fucnction call and executing the response (or exception raised) into a `Part` ready to be sent to the LLM again.

In [36]:
mk_fres_content(f_results(respt.function_calls))

<ul><li><code>parts</code>: <details open='true'><summary>parts[0]</summary><ul><li><code>function_response</code>: <ul><li><code>name</code>: add1</li><li><code>response</code>: <ul><li><b>result</b>: 646587811</li></ul></li></ul></li></ul></details></li><li><code>role</code>: tool</li></ul>

In [55]:
#| exports
@patch
def _call_tools(self: genai.models.Models, r):
    if r.function_calls:
        self.result.append(mk_fres_content(f_results(r.function_calls, ns=getattr(self, "_tools", None))))
   

If the response from an LLM is a function call, we want to make sure to actually call the functions, and append the results to the tool results.

In [56]:
#| exports

def prep_tools(tools, toolify_everything=False):
    funcs = [prep_tool(f, as_decl=toolify_everything) for f in tools if inspect.isfunction(f) or inspect.ismethod(f)]
    if toolify_everything: funcs = [types.Tool(function_declarations=[f]) for f in funcs]
    tools_ = [t for t in tools if isinstance(t, types.Tool)]
    class_tools = [types.Tool(function_declarations=[prep_tool(f, as_decl=True)]) for f in tools if inspect.isclass(f)]
    return funcs + tools_ + class_tools

Since we can pass several tools at once, and functions can be handled directly using the `automated function calling` from the SDK itself, in order to keep the API consistent with Claudette and Cosette, we need to add an intermediate layer to prepare the tools. Notice that this is defining each single function as a tool, while in principle we could group several into a tool (so that we can force the LLM to use a subset of tools).

In [57]:
a,b = 1232414,9415135
pr = f"What is {a}+{b}?"

c.models._tools=[add1]
c.models.post_cbs = [c.models._call_tools]

respt = g(c, pr, model=model, tools=[tool])
respt

<ul><li><code>add1(a=1232414, b=9415135)</code></li></ul>
<details><ul><li><code>automatic_function_calling_history</code>: </li><li><code>usage_metadata</code>: Cached: 0; In: 82; Out: 4; Total: 86</li><li><code>candidates</code>: <details open='true'><summary>candidates[0]</summary><ul><li><code>avg_logprobs</code>: 1.1253971024416387e-05</li><li><code>finish_reason</code>: FinishReason.STOP</li><li><code>content</code>: <ul><li><code>parts</code>: <details open='true'><summary>parts[0]</summary><ul><li><code>function_call</code>: <ul><li><code>args</code>: <ul><li><b>a</b>: 1232414</li>
<li><b>b</b>: 9415135</li></ul></li><li><code>name</code>: add1</li></ul></li></ul></details></li><li><code>role</code>: model</li></ul></li></ul></details></li><li><code>model_version</code>: gemini-2.0-flash</li></ul></details>

In [60]:
g(c, [pr] + c.result, model=model, tools=[tool])

1232414+9415135 is 10647549.<br />
<details><ul><li><code>automatic_function_calling_history</code>: </li><li><code>usage_metadata</code>: Cached: 0; In: 89; Out: 27; Total: 116</li><li><code>candidates</code>: <details open='true'><summary>candidates[0]</summary><ul><li><code>avg_logprobs</code>: -0.013651176735206886</li><li><code>finish_reason</code>: FinishReason.STOP</li><li><code>content</code>: <ul><li><code>parts</code>: <details open='true'><summary>parts[0]</summary><ul><li><code>text</code>: 1232414+9415135 is 10647549.
</li></ul></details></li><li><code>role</code>: model</li></ul></li></ul></details></li><li><code>model_version</code>: gemini-2.0-flash</li></ul></details>

## Automatic Function Calling

If instead of tools, we pass functions, we can rely on the SDK Automatic function calling, which essentially completes the whole tool loop by itself (checking for errors as well). The main drawback is that it is more limited in terms of function parameters types and it completely relies on the docstring of the function (although we turn the docments comments into a more complete docstring while prepping the function). In other words, when automatically building turning the function into a tool, the parameters descriptions are not set when using AFC. It is uncler how much this might affect the function calling performance

In [61]:
def sums(
    a:int,  # First number to sum 
    b=1 # Second number to sum
) -> int: # The sum of the inputs
    "Adds two numbers"
    print(f"Finding the sum of {a} and {b}")
    return a + b

a,b = 604542,6458932
pr = f"What is {a}+{b}?"
pr

'What is 604542+6458932?'

In [62]:
def mults(
    a:int,  # First thing to multiply
    b:int=1 # Second thing to multiply
) -> int: # The product of the inputs
    "Multiplies a * b."
    print(f"Finding the product of {a} and {b}")
    return a * b

pr = f'Calculate ({a}+{b})*2'
pr

'Calculate (604542+6458932)*2'

In [64]:
resp = g(c, pr, model=model, tools=prep_tools([sums, mults]))
resp

Finding the sum of 604542 and 6458932
Finding the product of 7063474 and 2


(604542+6458932)*2 = 14126948<br />
<details><ul><li><code>automatic_function_calling_history</code>: <details open='true'><summary>automatic_function_calling_history[0]</summary><ul><li><code>parts</code>: <details open='true'><summary>parts[0]</summary><ul><li><code>text</code>: Calculate (604542+6458932)*2</li></ul></details></li></ul></details>
<details open='true'><summary>automatic_function_calling_history[1]</summary><ul><li><code>parts</code>: <details open='true'><summary>parts[0]</summary><ul><li><code>function_call</code>: <ul><li><code>args</code>: <ul><li><b>b</b>: 6458932</li>
<li><b>a</b>: 604542</li></ul></li><li><code>name</code>: sums</li></ul></li></ul></details></li><li><code>role</code>: model</li></ul></details>
<details open='true'><summary>automatic_function_calling_history[2]</summary><ul><li><code>parts</code>: <details open='true'><summary>parts[0]</summary><ul><li><code>function_response</code>: <ul><li><code>name</code>: sums</li><li><code>response</code>: <ul><li><b>result</b>: 7063474</li></ul></li></ul></li></ul></details></li><li><code>role</code>: user</li></ul></details>
<details open='true'><summary>automatic_function_calling_history[3]</summary><ul><li><code>parts</code>: <details open='true'><summary>parts[0]</summary><ul><li><code>function_call</code>: <ul><li><code>args</code>: <ul><li><b>a</b>: 7063474</li>
<li><b>b</b>: 2</li></ul></li><li><code>name</code>: mults</li></ul></li></ul></details></li><li><code>role</code>: model</li></ul></details>
<details open='true'><summary>automatic_function_calling_history[4]</summary><ul><li><code>parts</code>: <details open='true'><summary>parts[0]</summary><ul><li><code>function_response</code>: <ul><li><code>name</code>: mults</li><li><code>response</code>: <ul><li><b>result</b>: 14126948</li></ul></li></ul></li></ul></details></li><li><code>role</code>: user</li></ul></details></li><li><code>usage_metadata</code>: Cached: 0; In: 104; Out: 28; Total: 132</li><li><code>candidates</code>: <details open='true'><summary>candidates[0]</summary><ul><li><code>avg_logprobs</code>: -0.0006584135283316885</li><li><code>finish_reason</code>: FinishReason.STOP</li><li><code>content</code>: <ul><li><code>parts</code>: <details open='true'><summary>parts[0]</summary><ul><li><code>text</code>: (604542+6458932)*2 = 14126948
</li></ul></details></li><li><code>role</code>: model</li></ul></li></ul></details></li><li><code>model_version</code>: gemini-2.0-flash</li></ul></details>

In [65]:
resp.automatic_function_calling_history

[UserContent(parts=[Part(video_metadata=None, thought=None, code_execution_result=None, executable_code=None, file_data=None, function_call=None, function_response=None, inline_data=None, text='Calculate (604542+6458932)*2')], role='user'),
 Content(parts=[Part(video_metadata=None, thought=None, code_execution_result=None, executable_code=None, file_data=None, function_call=FunctionCall(id=None, args={'b': 6458932, 'a': 604542}, name='sums'), function_response=None, inline_data=None, text=None)], role='model'),
 Content(parts=[Part(video_metadata=None, thought=None, code_execution_result=None, executable_code=None, file_data=None, function_call=None, function_response=FunctionResponse(id=None, name='sums', response={'result': 7063474}), inline_data=None, text=None)], role='user'),
 Content(parts=[Part(video_metadata=None, thought=None, code_execution_result=None, executable_code=None, file_data=None, function_call=FunctionCall(id=None, args={'a': 7063474, 'b': 2}, name='mults'), functi

In [66]:
g(c, "What is a good game to play with a dog on a rainy day?", model=model, tools=prep_tools([sums, mults]))

I am sorry, I am not able to help with that. I can only do addition and multiplication.
<details><ul><li><code>automatic_function_calling_history</code>: </li><li><code>usage_metadata</code>: Cached: 0; In: 89; Out: 21; Total: 110</li><li><code>candidates</code>: <details open='true'><summary>candidates[0]</summary><ul><li><code>avg_logprobs</code>: -0.19412753695533388</li><li><code>finish_reason</code>: FinishReason.STOP</li><li><code>content</code>: <ul><li><code>parts</code>: <details open='true'><summary>parts[0]</summary><ul><li><code>text</code>: I am sorry, I am not able to help with that. I can only do addition and multiplication.</li></ul></details></li><li><code>role</code>: model</li></ul></li></ul></details></li><li><code>model_version</code>: gemini-2.0-flash</li></ul></details>

Notice how if passing tools to the calls, Gemini assumes that its sole purpose is to call those tools.

In [68]:
g(c, "What is a good game to play with a dog on a rainy day?", model=model, tools=prep_tools([sums, mults]), tool_mode='NONE')

Rainy days can be tough for dogs and their owners! Here are some good indoor games to play with your dog on a rainy day, keeping both of you entertained and active:<br /><br />**Mental Stimulation Games:**<br /><br />*   **Hide-and-Seek (with treats or toys):** This is a classic! Have someone hold your dog while you hide a treat or toy in a visible but slightly challenging spot (under a blanket, behind a cushion). Release your dog and encourage them to "find it!" Gradually increase the difficulty as your dog gets better.<br />*   **Treat Puzzles & Interactive Toys:** Invest in some puzzle toys that require your dog to manipulate them to get a treat. These are great for tiring them out mentally. Examples include:<br />    *   Rolling treat balls<br />    *   Sliding puzzles<br />    *   Snuffle mats (where you hide treats in fabric folds)<br />*   **"Which Hand?" Game:** Hold a treat in one hand and close both fists. Let your dog sniff and paw at your hands to guess which one has the treat. Reward them when they choose correctly.<br />*   **Teaching New Tricks:** Rainy days are perfect for focusing on training. Teach your dog a new trick like "play dead," "shake," or "roll over." Keep training sessions short and positive, using lots of praise and treats.<br />*   **Name That Toy:** Teach your dog the names of their toys and then ask them to "get [toy name]". Start with just 2-3 toys and gradually add more.<br /><br />**Physical Games (with modifications for indoor space):**<br /><br />*   **Indoor Fetch (with a soft toy):** Use a soft toy or ball to play fetch in a hallway or open area. Keep throws short and controlled to avoid accidents.<br />*   **Tug-of-War:** A great way to burn energy and build a bond. Make sure to establish rules: you start and end the game, and the tug toy is yours when you say "drop it."<br />*   **Obstacle Course (using household items):** Get creative and use pillows, blankets, chairs, and tunnels to create a mini obstacle course. Guide your dog through it with treats and encouragement.<br />*   **Stair Climbs (if safe and appropriate):** If you have stairs, you can have your dog walk up and down them a few times (if they are physically able and your vet approves). This can be a good way to burn energy, but be careful not to overdo it.<br /><br />**Important Considerations:**<br /><br />*   **Your Dog's Age and Physical Condition:** Adjust the games based on your dog's age, breed, and any health issues. Avoid strenuous activities for puppies, senior dogs, or dogs with joint problems.<br />*   **Your Home's Safety:** Clear away any breakable items or hazards before starting a game.<br />*   **Keep it Positive:** Use positive reinforcement (treats, praise) to encourage your dog. Avoid punishment or scolding.<br />*   **Short and Frequent Sessions:** Keep the games short and frequent to keep your dog engaged and prevent boredom.<br />*   **Observe Your Dog:** Pay attention to your dog's body language and stop the game if they seem tired, stressed, or uninterested.<br /><br />**Example Rainy Day Schedule:**<br /><br />1.  **Morning:** Start with a short training session (10-15 minutes) to practice a new trick.<br />2.  **Mid-day:** Play a game of hide-and-seek with treats or use a puzzle toy.<br />3.  **Afternoon:** Engage in a game of tug-of-war or indoor fetch.<br />4.  **Evening:** Wind down with a snuffle mat filled with treats or a calming massage.<br /><br />By mixing up different activities, you can keep your dog entertained and happy on a rainy day!  Have fun!<br />
<details><ul><li><code>automatic_function_calling_history</code>: </li><li><code>usage_metadata</code>: Cached: 0; In: 15; Out: 801; Total: 816</li><li><code>candidates</code>: <details open='true'><summary>candidates[0]</summary><ul><li><code>avg_logprobs</code>: -0.325440100813924</li><li><code>finish_reason</code>: FinishReason.STOP</li><li><code>content</code>: <ul><li><code>parts</code>: <details open='true'><summary>parts[0]</summary><ul><li><code>text</code>: Rainy days can be tough for dogs and their owners! Here are some good indoor games to play with your dog on a rainy day, keeping both of you entertained and active:

**Mental Stimulation Games:**

*   **Hide-and-Seek (with treats or toys):** This is a classic! Have someone hold your dog while you hide a treat or toy in a visible but slightly challenging spot (under a blanket, behind a cushion). Release your dog and encourage them to "find it!" Gradually increase the difficulty as your dog gets better.
*   **Treat Puzzles & Interactive Toys:** Invest in some puzzle toys that require your dog to manipulate them to get a treat. These are great for tiring them out mentally. Examples include:
    *   Rolling treat balls
    *   Sliding puzzles
    *   Snuffle mats (where you hide treats in fabric folds)
*   **"Which Hand?" Game:** Hold a treat in one hand and close both fists. Let your dog sniff and paw at your hands to guess which one has the treat. Reward them when they choose correctly.
*   **Teaching New Tricks:** Rainy days are perfect for focusing on training. Teach your dog a new trick like "play dead," "shake," or "roll over." Keep training sessions short and positive, using lots of praise and treats.
*   **Name That Toy:** Teach your dog the names of their toys and then ask them to "get [toy name]". Start with just 2-3 toys and gradually add more.

**Physical Games (with modifications for indoor space):**

*   **Indoor Fetch (with a soft toy):** Use a soft toy or ball to play fetch in a hallway or open area. Keep throws short and controlled to avoid accidents.
*   **Tug-of-War:** A great way to burn energy and build a bond. Make sure to establish rules: you start and end the game, and the tug toy is yours when you say "drop it."
*   **Obstacle Course (using household items):** Get creative and use pillows, blankets, chairs, and tunnels to create a mini obstacle course. Guide your dog through it with treats and encouragement.
*   **Stair Climbs (if safe and appropriate):** If you have stairs, you can have your dog walk up and down them a few times (if they are physically able and your vet approves). This can be a good way to burn energy, but be careful not to overdo it.

**Important Considerations:**

*   **Your Dog's Age and Physical Condition:** Adjust the games based on your dog's age, breed, and any health issues. Avoid strenuous activities for puppies, senior dogs, or dogs with joint problems.
*   **Your Home's Safety:** Clear away any breakable items or hazards before starting a game.
*   **Keep it Positive:** Use positive reinforcement (treats, praise) to encourage your dog. Avoid punishment or scolding.
*   **Short and Frequent Sessions:** Keep the games short and frequent to keep your dog engaged and prevent boredom.
*   **Observe Your Dog:** Pay attention to your dog's body language and stop the game if they seem tired, stressed, or uninterested.

**Example Rainy Day Schedule:**

1.  **Morning:** Start with a short training session (10-15 minutes) to practice a new trick.
2.  **Mid-day:** Play a game of hide-and-seek with treats or use a puzzle toy.
3.  **Afternoon:** Engage in a game of tug-of-war or indoor fetch.
4.  **Evening:** Wind down with a snuffle mat filled with treats or a calming massage.

By mixing up different activities, you can keep your dog entertained and happy on a rainy day!  Have fun!
</li></ul></details></li><li><code>role</code>: model</li></ul></li></ul></details></li><li><code>model_version</code>: gemini-2.0-flash</li></ul></details>

## Structured calls

In [211]:
#| exports

@patch
def structured(self: genai.models.Models, inps, tool, model=None, **kwargs):
    self._tools=[tool]
    if not getattr(self, 'post_cbs', False): self.post_cbs = [self._call_tools]
    tools = prep_tools([tool])
    config = self._genconf(temp=0., use_afc=False, tools=tools, tool_mode="ANY", model=model, **kwargs)
    contents = mk_contents(inps, self)
    _ = self._gen(inps,  model, config, stream=False)
    return [nested_idx(ct, "function_response", "response", "result") for ct in nested_idx(self, "result_content", -1, "parts") or []]

@patch
def structured(self: genai.Client, inps, tool, model=None):
    return self.models.structured(inps, tool, model)

Defining the `structured` interface is essentially a matter of setting `tool_mode` to `ANY` (so that the LLM is forced to use the passed tool), and returning the function result`

In [212]:
c.structured(pr, add1, model=model)

[14126948]

In [240]:
class President:
    "Information about a president of the United States"
    def __init__(self, 
                first:str, # President first name.
                last:str, # President last name.
                spouse:str, # President's spouse name. 
                years_in_office:str, # President years in office, formatted as: {start_year}-{end_year}
                birth_year:int=0 # President year of birth (`0` if unknown). MANDATORY!
        ):
        assert re.match(r'\d{4}-\d{4}', years_in_office), "Invalid format: `years_in_office`: should be : '{start_year}-{end_year}'"
        store_attr()

    __repr__ = basic_repr('first, last, spouse, years_in_office, birth_year')




Gemini is very picky about tool usage and it tends to be quite lazy. In the example above, removing the `MANDATORY` comment, for the last argument will cause Gemini to skip the spouse name and/or ignore the birth year. 

In [242]:
c.structured("Key details about the first 10 Presidents of the United States", President, model=models[0])

[President(first='George', last='Washington', spouse='Martha Dandridge Custis', years_in_office='1789-1797', birth_year=1732),
 President(first='John', last='Adams', spouse='Abigail Smith', years_in_office='1797-1801', birth_year=1735),
 President(first='Thomas', last='Jefferson', spouse='Martha Wayles Skelton', years_in_office='1801-1809', birth_year=1743),
 President(first='James', last='Madison', spouse='Dolley Payne Todd', years_in_office='1809-1817', birth_year=1751),
 President(first='James', last='Monroe', spouse='Elizabeth Kortright', years_in_office='1817-1825', birth_year=1758),
 President(first='John', last='Quincy Adams', spouse='Louisa Catherine Johnson', years_in_office='1825-1829', birth_year=1767),
 President(first='Andrew', last='Jackson', spouse='Rachel Donelson', years_in_office='1829-1837', birth_year=1767),
 President(first='Martin', last='Van Buren', spouse='Hannah Hoes', years_in_office='1837-1841', birth_year=1782),
 President(first='William', last='Henry Harris

**TODO:** Right now we are not using the structured output capabilities of the Genai API, which allow to pas Pydantic models or JSON schemas to structure the output of the response. 

In [243]:
#|hide
#|eval: false

import nbdev; nbdev.nbdev_export()